<a href="https://colab.research.google.com/github/abduyea/Career-Trends-Analyzer/blob/main/notebooks/data_ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Abdulfetah Adem
##  Spencer K

Core setup + load all raw tables + master

Setup + imports in data_ingestion.ipynb

In [ ]:
from pathlib import Path

SRC_DIR = Path("/content/drive/MyDrive/Career-Trends-Analyzer/src")

data_loader_code = """
from __future__ import annotations

from pathlib import Path
from typing import Dict

import pandas as pd

from .config import RAW_DIR

try:
    from google.colab import drive
except ImportError:
    drive = None


def mount_drive() -> None:
    # mount only in Colab
    if drive:
        drive.mount("/content/drive", force_remount=False)


def _load_csv(path: Path) -> pd.DataFrame:
    # load single csv
    if not path.is_file():
        raise FileNotFoundError(f"CSV not found: {path}")
    return pd.read_csv(path)


def load_postings() -> pd.DataFrame:
    # postings.csv
    return _load_csv(RAW_DIR / "postings.csv")


def _load_folder(name: str) -> Dict[str, pd.DataFrame]:
    # load all csvs in raw/<name>
    folder = RAW_DIR / name
    if not folder.is_dir():
        return {}
    return {p.stem: pd.read_csv(p) for p in folder.glob("*.csv")}


def load_companies() -> Dict[str, pd.DataFrame]:
    # raw/companies
    return _load_folder("companies")


def load_jobs() -> Dict[str, pd.DataFrame]:
    # raw/jobs
    return _load_folder("jobs")


def load_mappings() -> Dict[str, pd.DataFrame]:
    # raw/mappings
    return _load_folder("mappings")


def build_master(
    postings: pd.DataFrame,
    companies: Dict[str, pd.DataFrame],
    jobs: Dict[str, pd.DataFrame],
    mappings: Dict[str, pd.DataFrame],
) -> pd.DataFrame:
    # safe merges
    df = postings.copy()

    c = companies.get("companies")
    if c is not None and "company_id" in df and "company_id" in c:
        df = df.merge(c, on="company_id", how="left", suffixes=("", "_company"))

    s = jobs.get("salaries")
    if s is not None and "job_id" in df and "job_id" in s:
        df = df.merge(s, on="job_id", how="left", suffixes=("", "_salary"))

    return df
"""

path = SRC_DIR / "data_loader.py"
path.write_text(data_loader_code.strip() + "\n", encoding="utf-8")
print("Rewrote:", path)


Rewrote: /content/drive/MyDrive/Career-Trends-Analyzer/src/data_loader.py


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Career-Trends-Analyzer").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import (
    mount_drive,
    load_postings,
    load_companies,
    load_jobs,
    load_mappings,
    build_master,
)
from src.config import ROOT_DIR, RAW_DIR
from src.utils import peek, missing_summary, summarize_tables

print("Imports OK. ROOT_DIR:", ROOT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Imports OK. ROOT_DIR: /content/drive/MyDrive/Career-Trends-Analyzer


## Data laoding

In [ ]:
mount_drive()

postings = load_postings()
companies = load_companies()
jobs = load_jobs()
mappings = load_mappings()

master = build_master(postings, companies, jobs, mappings)
peek(master)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(123849, 47)


,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,zip_code_company,address,url,salary_id,max_salary_salary,med_salary_salary,min_salary_salary,pay_period_salary,currency_salary,compensation_type_salary
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,...,07302,242 Tenth Street,https://www.linkedin.com/company/corcoran-sawy...,18531.0,20.0,NaN,17.0,HOURLY,USD,BASE_SALARY
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,...,NaN,NaN,NaN,8059.0,50.0,NaN,30.0,HOURLY,USD,BASE_SALARY
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,...,45227,6880 Wooster Pike,https://www.linkedin.com/company/the-national-...,14949.0,65000.0,NaN,45000.0,YEARLY,USD,BASE_SALARY
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,175000.0,YEARLY,"New Hyde Park, NY",766262.0,16.0,NaN,...,11042,3 Dakota Drive,https://www.linkedin.com/company/abrams-fenste...,11204.0,175000.0,NaN,140000.0,YEARLY,USD,BASE_SALARY
4,35982263,NaN,Service Technician,Looking for HVAC service tech with experience ...,80000.0,YEARLY,"Burlington, IA",NaN,3.0,NaN,...,NaN,NaN,NaN,20809.0,80000.0,NaN,60000.0,YEARLY,USD,BASE_SALARY
